# Motor - finger tapping features + Balanced RF (3 classes)

**Spec:** PLAN.md B4. **Runs on:** Kaggle. Not run yet.

No labelled tapping data is available to us (FINDINGS 2). Cell 2 checks that the feature code works on a made-up session; **it is not data and shows no performance**. Cells 3-4 only run if you ever obtain real files, laid out as `DATA/<subject_id>.csv` (PsychoPy output of the reference task) plus `DATA/labels.csv` with columns `subject,group` where group is CN, aMCI or AD.

Upload the repo's `common/` and `motor/` folders. Set `REPO`.

In [ ]:
!pip -q install imbalanced-learn
import sys, os, numpy as np, pandas as pd
REPO = '/kaggle/working/Alzheimers'          # folder containing common/ and motor/
DATA = '/kaggle/input/finger-tapping'        # EDIT if you ever obtain data
sys.path.insert(0, REPO)
from motor.features import session_features, FEATURE_NAMES
from motor.loader import trials_from_csv
from motor.cv import evaluate, CLASSES

## 2. Code check on a made-up session (not data)

In [ ]:
r = [(0.4 + 0.25 * i, 'rshift') for i in range(38)]
d = [(t, k) for i in range(18) for t, k in ((0.5 + 0.5 * i, 'lshift'), (0.52 + 0.5 * i, 'rshift'))]
a = [(0.5 + 0.3 * i, 'lshift' if i % 2 == 0 else 'rshift') for i in range(30)]
s = session_features({'right': [r, r, r], 'dual': [d, d, d], 'alt': [a, a, a]})
print(len(s), 'features'); pd.Series(s).round(3)

## 3. Load real data (only if present)

In [ ]:
lab = pd.read_csv(f'{DATA}/labels.csv')
rows = [session_features(trials_from_csv(f'{DATA}/{s}.csv')) for s in lab.subject]
X = pd.DataFrame(rows)[FEATURE_NAMES].values
y = lab.group.map({c: i for i, c in enumerate(CLASSES)}).values
print(X.shape, np.bincount(y), 'missing cells:', int(np.isnan(X).sum()))

## 4. Cross-validated result
Report only what this prints. Not comparable to the paper's 70% / 0.76 (different protocol).

In [ ]:
for r in evaluate(X, y): print(r)